In [17]:
import pandas as pd
from tqdm import tqdm
import pickle

In [18]:
names=["tama_augustus","tama_liftoff"]
runs=["../01_preparation/all_runs/tama_augustus_sqanti.pkl","../01_preparation/all_runs/tama_liftoff_sqanti.pkl"]

refs=pd.DataFrame()
for name, run in zip(names,runs):
    # load full df
    ref=pd.read_pickle(f"{run}")
    
    # add new column
    ref["cat_assign"]="None"
    # and fill new colum based on 4 categories
    ref.loc[(ref["structural_category"] != "antisense") & (ref["structural_category"] != "fusion") & (ref["structural_category"] != "intergenic"), "cat_assign"]="one2one"
    ref.loc[(ref["structural_category"] == "antisense"), "cat_assign"]="antisense"
    ref.loc[(ref["structural_category"] == "fusion"), "cat_assign"]="fusion"
    ref.loc[(ref["structural_category"] == "intergenic") | (ref["structural_category"] == "genic_intron"), "cat_assign"]="novel"
    
    # in some cases, transcripts were reported as one2one even though the reads are spanning two genes.
    # In these cases, we have "fusion reads", but splicing results in perfect cds for one gene.
    # filter them and add replace the assignment
    ref["association"] = ref["associated_gene"].str.split("_").str.len()
    ref.loc[(ref["cat_assign"]=="one2one")&(ref["association"]>1), "cat_assign"] = "fusion"
    
    ref = ref.loc[(ref["cat_assign"]=="antisense")|(ref["cat_assign"]=="novel")]    
    ref["run"]=name
    refs=pd.concat([refs,ref])

In [3]:
#Do second run with only coding ones
#refs = refs.loc[~(refs["Sqanti_ORF_seq"].isna())]

In [4]:
numbers_all=pd.DataFrame()
novel_intersect={}
for acc in refs["accession"].unique().tolist():
    df=refs.loc[refs["accession"]==acc]
    acc_df={}
    ### General
    a_novel=df.loc[(df["run"]=="tama_augustus")]
    l_novel=df.loc[(df["run"]=="tama_liftoff")]
    ## augustus
    #n_isoforms
    acc_df["aug_isoforms"]=len(a_novel["isoform"].tolist())
    #print(len(a_novel["isoform"].tolist()))
    #n_genes
    acc_df["aug_genes"]=len(a_novel.groupby("gene"))
    #n_noncoding
    acc_df["aug_noncoding"]=len(a_novel.loc[a_novel["Sqanti_ORF_seq"].isna()])

    ## liftoff
    #n_isoforms
    acc_df["lift_isoforms"]=len(l_novel["isoform"].tolist())
    #n_genes
    acc_df["lift_genes"]=len(l_novel.groupby("gene"))
    #n_noncoding
    acc_df["lift_noncoding"]=len(l_novel.loc[l_novel["Sqanti_ORF_seq"].isna()])

    ### Indepth
    a_novel_anti=df.loc[(df["run"]=="tama_augustus")&(df["cat_assign"]=="antisense")]
    a_novel_nov=df.loc[(df["run"]=="tama_augustus")&(df["cat_assign"]=="novel")]
    l_novel_anti=df.loc[(df["run"]=="tama_liftoff")&(df["cat_assign"]=="antisense")]
    l_novel_nov=df.loc[(df["run"]=="tama_liftoff")&(df["cat_assign"]=="novel")]
    ## augustus
    # n_isoforms
    acc_df["aug_anti_isoforms"]=len(a_novel_anti["isoform"].tolist())
    # n_genes
    acc_df["aug_anti_genes"]=len(a_novel_anti.groupby("gene"))
    # n_isoforms
    acc_df["aug_novel_isoforms"]=len(a_novel_nov["isoform"].tolist())
    # n_genes
    acc_df["aug_novel_genes"]=len(a_novel_nov.groupby("gene"))

    ## liftoff
    #n_isoforms
    acc_df["lift_anti_isoforms"]=len(l_novel_anti["isoform"].tolist())
    #n_genes
    acc_df["lift_anti_genes"]=len(l_novel_anti.groupby("gene"))
    #n_isoforms
    acc_df["lift_novel_isoforms"]=len(l_novel_nov["isoform"].tolist())
    #n_genes
    acc_df["lift_novel_genes"]=len(l_novel_nov.groupby("gene"))
    
    # get unique novelties
    aug=set(a_novel["isoform"].tolist())
    lif=set(l_novel["isoform"].tolist())
    intersection=list(aug.intersection(lif))
    # unique isoforms
    acc_df["unique_isoforms"]=len(intersection)
    # unique genes
    acc_df["unique_genes"]=len(df.loc[df["isoform"].isin(intersection)].groupby("gene"))
    
    # combine
    acc_df["accession"]=acc
    
    acc_df=pd.DataFrame([acc_df])
    numbers_all=pd.concat([numbers_all,acc_df])
    
    # extend
    novel_intersect[acc]=intersection

In [5]:
numbers_all.to_csv("numbers_all.tsv", sep="\t", index=False)

In [16]:
with open('intersection.pickle', 'wb') as handle:
    pickle.dump(novel_intersect, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Prot sequences (filter afterwards)

In [6]:
def read(file_name):
    try:
        fasta = open(file_name, 'r')
        fasta_pairs = {}
        header = ''
        # check every line for a new header
        # if so, set as key and set non header lines as value
        for line in fasta:
            if line[0] == '>':
                header = line.strip('\n')
                fasta_pairs[header] = ''
            else:
                fasta_pairs[header] += line.strip('\n')
        # return dict of fasta entries
        return fasta_pairs
    except IOError:
        print("File \"{}\" could not be opened.".format(file_name))

In [7]:
for acc in novel_intersect.keys():
    fasta = read(f"../../output/v2/collapse_tama/{acc}_transdecoder/{acc}_transcripts.fasta.transdecoder.pep")
    
    f_out=[]
    for g in fasta.keys():
        name = g.replace(">","").split(".")[0] +"."+ g.replace(">","").split(".")[1]
        if name in novel_intersect[f"{acc}"]:
            f_out.append(g)
            
    ofile = open(f"input/{acc}_novel.prot.fasta", "w")

    for g in f_out:
        ofile.write(g + "\n" + fasta[g].replace("*","") + "\n")

    #do not forget to close it
    ofile.close()

In [11]:
#Do second run with only coding ones
refs_c = refs.loc[~(refs["Sqanti_ORF_seq"].isna())]

In [12]:
numbers_all=pd.DataFrame()
novel_intersect={}
for acc in refs_c["accession"].unique().tolist():
    df=refs_c.loc[refs_c["accession"]==acc]
    acc_df={}
    ### General
    a_novel=df.loc[(df["run"]=="tama_augustus")]
    l_novel=df.loc[(df["run"]=="tama_liftoff")]
    ## augustus
    #n_isoforms
    acc_df["aug_isoforms"]=len(a_novel["isoform"].tolist())
    #print(len(a_novel["isoform"].tolist()))
    #n_genes
    acc_df["aug_genes"]=len(a_novel.groupby("gene"))
    #n_noncoding
    acc_df["aug_noncoding"]=len(a_novel.loc[a_novel["Sqanti_ORF_seq"].isna()])

    ## liftoff
    #n_isoforms
    acc_df["lift_isoforms"]=len(l_novel["isoform"].tolist())
    #n_genes
    acc_df["lift_genes"]=len(l_novel.groupby("gene"))
    #n_noncoding
    acc_df["lift_noncoding"]=len(l_novel.loc[l_novel["Sqanti_ORF_seq"].isna()])

    ### Indepth
    a_novel_anti=df.loc[(df["run"]=="tama_augustus")&(df["cat_assign"]=="antisense")]
    a_novel_nov=df.loc[(df["run"]=="tama_augustus")&(df["cat_assign"]=="novel")]
    l_novel_anti=df.loc[(df["run"]=="tama_liftoff")&(df["cat_assign"]=="antisense")]
    l_novel_nov=df.loc[(df["run"]=="tama_liftoff")&(df["cat_assign"]=="novel")]
    ## augustus
    # n_isoforms
    acc_df["aug_anti_isoforms"]=len(a_novel_anti["isoform"].tolist())
    # n_genes
    acc_df["aug_anti_genes"]=len(a_novel_anti.groupby("gene"))
    # n_isoforms
    acc_df["aug_novel_isoforms"]=len(a_novel_nov["isoform"].tolist())
    # n_genes
    acc_df["aug_novel_genes"]=len(a_novel_nov.groupby("gene"))

    ## liftoff
    #n_isoforms
    acc_df["lift_anti_isoforms"]=len(l_novel_anti["isoform"].tolist())
    #n_genes
    acc_df["lift_anti_genes"]=len(l_novel_anti.groupby("gene"))
    #n_isoforms
    acc_df["lift_novel_isoforms"]=len(l_novel_nov["isoform"].tolist())
    #n_genes
    acc_df["lift_novel_genes"]=len(l_novel_nov.groupby("gene"))
    
    # get unique novelties
    aug=set(a_novel["isoform"].tolist())
    lif=set(l_novel["isoform"].tolist())
    intersection=list(aug.intersection(lif))
    # unique isoforms
    acc_df["unique_isoforms"]=len(intersection)
    # unique genes
    acc_df["unique_genes"]=len(df.loc[df["isoform"].isin(intersection)].groupby("gene"))
    
    # combine
    acc_df["accession"]=acc
    
    acc_df=pd.DataFrame([acc_df])
    numbers_all=pd.concat([numbers_all,acc_df])
    
    # extend
    novel_intersect[acc]=intersection

In [13]:
numbers_all.to_csv("numbers_all_coding.tsv", sep="\t", index=False)